# D20A — Validação do backend Python `differences==0.3.0`

## tl;dr

O backend recuperou o ATT(g,t) do cenário limpo com 30 seeds (bias médio **0,047**, MAE **0,221**, RMSE **0,276**, correlação **0,995**) e cobertura simultânea observada de **100%** em 250 células de 10 seeds com 99 repetições de bootstrap. Não houve falha de convergência.

A validação é suficiente para dados sintéticos balanceados, mas **não libera a primeira estimação real**: remover Cabo Frio/2009 torna o painel desbalanceado e elimina Cabo Frio de todas as células pós-tratamento da coorte 2010 sob `base_period='universal'`. A decisão metodológica permanece bloqueada. Nenhum efeito real foi estimado.

## Contexto e métodos

A D20A substitui integralmente a proposta R por uma auditoria empírica e estrutural do backend Python. Reutiliza `src/simula_did_escalonado.py`; não cria DGP paralelo e não altera a D15.

### Premissas-chave

- coorte `g` é o primeiro ano tratado; `k=t-g`;
- controles principais são exclusivamente never-treated;
- `anticipation=0`, `base_period='universal'`, `est_method='dr'`;
- cluster por município é automático com `cluster_var=None`;
- painel desbalanceado deve usar `as_repeated_cross_section=False`;
- toda função de fit do projeto recusa dados não sintéticos; a D15 entra apenas em dry-run sem `.fit()`.

In [1]:
from pathlib import Path
import json
import sys
import tempfile

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import simula_did_escalonado as simulador
import valida_backend_differences as d20a
from visualizacao_ipt import CORES_IPT, aplicar_tema_ipt, salvar_figura_ipt

OUT_DIAG = ROOT / 'outputs' / 'diagnostics'
OUT_FIG = ROOT / 'outputs' / 'figures'
D15 = ROOT / 'data' / 'processed' / 'amostra_causal_cempre_2007_2019.parquet'
OUT_DIAG.mkdir(parents=True, exist_ok=True)

## API instalada e contrato operacional

A tabela abaixo vem da classe efetivamente importada na `.venv`, não de analogia com o pacote R.

In [2]:
api = d20a.auditar_api_differences()
api_tabela = pd.DataFrame({'item': api.keys(), 'valor': [str(v) for v in api.values()]})
api_tabela.to_csv(OUT_DIAG / 'D20A_api_differences.csv', index=False)
display(api_tabela)

,item,valor
0,versao,0.3.0
1,classe,differences.ATTgt
2,dataframe,pandas.DataFrame com MultiIndex unidade-tempo
3,indice,"dois níveis: entidade, tempo"
4,coorte,primeiro período tratado
5,never_treated_api,coorte_nula
6,base_period_opcoes,"('varying', 'universal')"
7,anticipation_default,0
8,control_group_default,never_treated
9,est_method_default,reg


### Coorte, período-base e controles

`universal` fixa `g-1` como base e materializa `k=-1` com ATT zero. `varying` usa diferenças sequenciais nos leads e não normaliza `k=-1`. Never-treated são representados por coorte nula; coortes futuras não entram no controle principal.

In [3]:
dgp_base = simulador.criar_cenario_limpo(seed=2012, n_unidades=120)
estimativa_universal = d20a.estimar_sintetico(dgp_base.painel, base_period='universal')
estimativa_varying = d20a.estimar_sintetico(dgp_base.painel, base_period='varying')
base_period_tabela = pd.concat([
    estimativa_universal['event'].assign(base_period='universal'),
    estimativa_varying['event'].assign(base_period='varying'),
]).loc[lambda x: x['event_time'].between(-3, 2)]
base_period_tabela.to_csv(OUT_DIAG / 'D20A_base_period.csv', index=False)
display(base_period_tabela[['base_period', 'event_time', 'att', 'std_error']])

,base_period,event_time,att,std_error
3,universal,-3,0.000075,0.192121
4,universal,-2,0.207218,0.187796
5,universal,-1,0.000000,NaN
6,universal,0,3.875984,0.281772
7,universal,1,5.097201,0.269440
8,universal,2,5.957085,0.264957
2,varying,-3,-0.035925,0.230455
3,varying,-2,0.207143,0.203688
4,varying,-1,-0.207218,0.187796
5,varying,0,3.875984,0.281772


## Resultados sintéticos

Os cinco cenários D19 são executados sem modificar os geradores. O cenário de violação de tendências paralelas é um controle negativo: espera-se viés, não aprovação artificial.

In [4]:
cenarios = d20a.executar_cenarios_d19(n_unidades=240)
for nome, tabela in cenarios.items():
    tabela.to_csv(OUT_DIAG / f'D20A_cenarios_{nome}.csv', index=False)
display(cenarios['resumo'].round(4))

,cenario,n_celulas,bias_medio,mae,rmse,correlacao,cobertura,n_celulas_falhas,parallel_trends_verdadeira,overlap_fraco,n_coorte_2013
0,LIMPO,25,-0.1356,0.1682,0.2119,0.9983,1.00,0,True,False,48
1,HETEROGENEIDADE_FORTE,25,0.0691,0.3458,0.3848,0.9977,0.84,0,True,False,48
2,VIOLACAO_PARALLEL_TRENDS,25,3.7132,3.7132,4.1754,0.9693,0.00,0,False,False,48
3,OVERLAP_FRACO,25,-0.1785,0.2346,0.2805,0.9968,0.88,0,True,True,48
4,COORTE_2013_N2,25,-0.1801,0.3122,0.4638,0.9875,0.92,0,True,False,2


In [5]:
limpo = cenarios['att_gt_comparacao'].loc[lambda x: x['cenario'].eq('LIMPO')].copy()
limpo['event_time'] = limpo['ano'] - limpo['coorte']
curva = limpo.groupby('event_time', as_index=False).agg(
    estimado=('att', 'mean'), verdadeiro=('att_gt_verdadeiro', 'mean')
)
fig_att = go.Figure()
fig_att.add_trace(go.Scatter(x=curva['event_time'], y=curva['verdadeiro'], mode='lines+markers', name='Verdade DGP', line={'color': CORES_IPT['AZUL_ESCURO'], 'dash': 'dash'}))
fig_att.add_trace(go.Scatter(x=curva['event_time'], y=curva['estimado'], mode='lines+markers', name='ATT estimado', line={'color': CORES_IPT['AZUL_PRINCIPAL']}))
fig_att.add_hline(y=0, line_color=CORES_IPT['CINZA_GRADE'])
fig_att.update_xaxes(title='Tempo relativo ao tratamento (k)')
fig_att.update_yaxes(title='ATT sintético (unidades de y)')
aplicar_tema_ipt(fig_att, titulo='D20A — ATT dinâmico estimado versus verdade conhecida')
salvar_figura_ipt(fig_att, OUT_FIG / 'D20A_att_dinamico_limpo.png')
fig_att.show()

## Monte Carlo predefinido

Antes dos resultados foram fixados 30 seeds para erro pontual e 10 seeds para cobertura, com 99 repetições do multiplier bootstrap por seed. A cobertura refere-se às bandas simultâneas do ATT(g,t).

In [6]:
%%capture
monte_carlo = d20a.executar_monte_carlo(
    seeds_ponto=list(range(23001, 23031)),
    seeds_bootstrap=list(range(24001, 24011)),
    n_unidades=240,
    boot_iterations=99,
)
monte_carlo['por_seed_ponto'].to_csv(OUT_DIAG / 'D20A_monte_carlo_por_seed_ponto.csv', index=False)
monte_carlo['por_seed_bootstrap'].to_csv(OUT_DIAG / 'D20A_monte_carlo_por_seed_bootstrap.csv', index=False)
monte_carlo['comparacoes_ponto'].to_csv(OUT_DIAG / 'D20A_monte_carlo_att_gt.csv', index=False)
monte_carlo['comparacoes_bootstrap'].to_csv(OUT_DIAG / 'D20A_monte_carlo_cobertura.csv', index=False)
pd.DataFrame([monte_carlo['resumo']]).to_csv(OUT_DIAG / 'D20A_monte_carlo_resumo.csv', index=False)
(OUT_DIAG / 'D20A_monte_carlo_configuracao.json').write_text(json.dumps(monte_carlo['configuracao'], indent=2), encoding='utf-8')

In [7]:
display(pd.DataFrame([monte_carlo['resumo']]).round(4))
fig_mc = go.Figure(go.Histogram(
    x=monte_carlo['por_seed_ponto']['bias_medio'],
    marker_color=CORES_IPT['AZUL_PRINCIPAL'],
    nbinsx=10,
))
fig_mc.add_vline(x=0, line_color=CORES_IPT['AZUL_ESCURO'], line_dash='dash')
fig_mc.update_xaxes(title='Bias médio por seed')
fig_mc.update_yaxes(title='Número de seeds')
aplicar_tema_ipt(fig_mc, titulo='D20A — Distribuição do bias no Monte Carlo limpo')
salvar_figura_ipt(fig_mc, OUT_FIG / 'D20A_monte_carlo_bias.png')
fig_mc.show()

,bias_medio,mae,rmse,correlacao,cobertura,n_celulas_ponto,n_celulas_cobertura,n_falhas_convergencia,n_seeds_ponto,n_seeds_bootstrap
0,0.0473,0.2207,0.2762,0.9949,1.0,750,250,0,30,10


## Inferência e limitação da referência

A versão 0.3.0 oferece multiplier bootstrap e bandas simultâneas. Com `universal`, incluir a célula determinística `k=-1` (variância zero) produz bandas `NaN` para todo o event-study. A camada D20A usa a API pública `filter_gt` para retirar `k=-1` apenas do bootstrap e o recoloca como referência zero na apresentação. Por depender desse workaround, o gate de inferência é **PARCIAL**.

## Cabo Frio e painel desbalanceado

No backend, um painel desbalanceado vira repeated cross-section por padrão. A D20A força modo painel com `as_repeated_cross_section=False`, que forma amostras 2×2 completas por célula. Como Cabo Frio/2009 é `g-1` para a coorte 2010, suas demais linhas permanecem no objeto, mas o município não participa de nenhum ATT pós-tratamento da coorte sob `universal`. Isso altera a amostra efetiva e exige decisão metodológica humana.

In [8]:
dgp_cabo = simulador.criar_cenario_limpo(seed=2030, n_unidades=80)
unidade_cabo = dgp_cabo.painel.loc[dgp_cabo.painel['coorte_g'].eq(2010), 'id'].iloc[0]
painel_cabo = dgp_cabo.painel.loc[~(dgp_cabo.painel['id'].eq(unidade_cabo) & dgp_cabo.painel['ano'].eq(2009))]
with tempfile.TemporaryDirectory() as pasta:
    estimacao_cabo = d20a.estimar_sintetico(painel_cabo, as_repeated_cross_section=False, files_path=pasta)
    celula_cabo = pd.read_parquet(Path(pasta) / 'cohort-2010_base_period-2009_time-2011.parquet')
codigo_interno = sorted(dgp_cabo.painel['id'].unique()).index(unidade_cabo)
auditoria_cabo = pd.DataFrame([{
    'unidade': unidade_cabo,
    'linhas_preservadas_no_objeto': estimacao_cabo['metadados']['linhas_unidade_desbalanceada'],
    'painel_balanceado': estimacao_cabo['metadados']['painel_balanceado'],
    'como_repeated_cross_section': estimacao_cabo['metadados']['como_repeated_cross_section'],
    'entra_ATT_2010_2011': codigo_interno in set(celula_cabo['id']),
    'decisao': 'BLOQUEADO_DECISAO',
}])
auditoria_cabo.to_csv(OUT_DIAG / 'D20A_cabo_frio_sintetico.csv', index=False)
display(auditoria_cabo)

,unidade,linhas_preservadas_no_objeto,painel_balanceado,como_repeated_cross_section,entra_ATT_2010_2011,decisao
0,S000002,12,False,False,False,BLOQUEADO_DECISAO


## Dry-run real sem fit

A D15 é lida, validada, copiada, filtrada somente pela máscara já congelada e instanciada na API. O módulo não expõe caminho de fit real.

In [9]:
dados_reais_api, auditoria_real = d20a.executar_dry_run_real_sem_fit(D15)
pd.DataFrame([auditoria_real]).to_csv(OUT_DIAG / 'D20A_dry_run_real.csv', index=False)
display(pd.DataFrame([auditoria_real]))

,n_linhas,n_elegiveis,n_unidades,n_tratados,n_controles,contagem_coortes,anos,painel_base_balanceado,chave_unica,anos_inelegiveis_cabo_frio,...,n_linhas_backend,n_linhas_cabo_frio,coorte_cabo_frio,painel_balanceado_backend,never_treated_representado_por_nulo,objeto_attgt_instanciado,fit_real_bloqueado,sha256_antes,sha256_depois,byte_identical
0,66196,66195,5092,129,4963,"{2009: 21, 2010: 27, 2011: 66, 2012: 13, 2013: 2}","[2007, 2008, 2009, 2010, 2011, 2012, 2013, 201...",True,True,[2009],...,66195,12,2010,False,True,True,True,7c24c01b569ed6d10f773d00599b5010e1607696f46d71...,7c24c01b569ed6d10f773d00599b5010e1607696f46d71...,True


## Gates e takeaways

O backend sintético está validado, mas inferência é parcial pelo workaround de `k=-1`, e o uso real permanece bloqueado pela incompatibilidade Cabo Frio × painel balanceado. Isso não reprova o desenho causal nem estima efeito real; apenas impede avançar automaticamente.

In [10]:
gates = d20a.gates_d20a(monte_carlo_validado=True, bootstrap_validado=True, dry_run_validado=True)
gates_tabela = pd.DataFrame({'gate': gates.keys(), 'valor': gates.values()})
gates_tabela.to_csv(OUT_DIAG / 'D20A_gates.csv', index=False)
display(gates_tabela)

,gate,valor
0,D20A_PYTHON_DIFERENCES_0_3_0_DISPONIVEL,SIM
1,D20A_BACKEND_CALLAWAY_SANTANNA_PYTHON_VALIDADO,SIM
2,D20A_CONVENCAO_COORTE_VALIDADA,SIM
3,D20A_BASE_PERIOD_VALIDADO,SIM
4,D20A_NEVER_TREATED_VALIDADO,SIM
5,D20A_ATT_GT_SINTETICO_VALIDADO,SIM
6,D20A_MONTE_CARLO_VALIDADO,SIM
7,D20A_EVENT_STUDY_SINTETICO_VALIDADO,SIM
8,D20A_INFERENCIA_VALIDADA,PARCIAL
9,D20A_PAINEL_DESBALANCEADO_VALIDADO,SIM


# D20B — Fechamento pré-estimação: Cabo Frio + inferência

A D20B distingue a população documental D15 (**129 tratados**) da população principal estimável (**128 tratados**). Cabo Frio permanece integralmente na D15, mas é retirado da cópia de estimação porque seu `g-1=2009` é institucionalmente inelegível. Nenhum efeito real é estimado.

## View principal estimável

A exclusão é explícita, integral e restrita à cópia derivada. Não há imputação, mudança de coorte, repeated cross-section nem alteração da D15.

In [11]:
VIEW_D20B = ROOT / 'data' / 'processed' / 'amostra_principal_estimavel_d20b_2007_2019.parquet'
view_d20b, auditoria_view_d20b = d20a.materializar_view_principal_estimavel(D15, VIEW_D20B)
auditoria_view_tabela = pd.DataFrame([auditoria_view_d20b])
auditoria_view_tabela['contagem_coortes'] = auditoria_view_tabela['contagem_coortes'].map(json.dumps)
auditoria_view_tabela.to_csv(OUT_DIAG / 'D20B_view_principal_estimavel.csv', index=False)
display(auditoria_view_tabela.T.rename(columns={0: 'valor'}))

,valor
POPULACAO_D15_TRATADOS,129
POPULACAO_PRINCIPAL_ESTIMAVEL,128
n_controles,4963
n_municipios,5091
n_linhas,66183
painel_balanceado_2007_2019,True
codigo_5003900_ausente,True
cabo_frio_ausente_view,True
cabo_frio_presente_d15,True
contagem_coortes,"{""2009"": 21, ""2010"": 26, ""2011"": 66, ""2012"": 1..."


## Invariância do workaround `k=-1`

O teste A usa `base_period='universal'` sem bootstrap ou filtro. O teste B remove `k=-1` antes do multiplier bootstrap e o reinsere somente como zero de referência. A validação deve bloquear se qualquer estimativa pontual não referencial mudar.

In [12]:
painel_workaround = simulador.criar_cenario_limpo(seed=25001, n_unidades=80).painel
validacao_workaround = d20a.validar_workaround_k_menos_1(
    painel_workaround, boot_iterations=99, random_state=20260924, n_jobs=1
)
invariancia_tabela = pd.DataFrame([
    {
        'componente': componente,
        'invariante': status,
        'max_abs_diff': validacao_workaround['max_abs_diff'].get(componente, 0.0),
    }
    for componente, status in validacao_workaround['invariantes'].items()
])
invariancia_tabela.to_csv(OUT_DIAG / 'D20B_workaround_k_menos_1.csv', index=False)
for componente, tabela in validacao_workaround['comparacoes'].items():
    tabela.to_csv(OUT_DIAG / f'D20B_invariancia_{componente}.csv', index=False)
display(invariancia_tabela)

Bootstrap                       0%|                    | [00:00<?, ?it/s]

Bootstrap                     100%|████████████████████| [00:00<00:00, 13000.91it/s]

Bootstrap                       0%|                    | [00:00<?, ?it/s]

Bootstrap                     100%|████████████████████| [00:00<00:00, 16505.79it/s]

Bootstrap                       0%|                    | [00:00<?, ?it/s]

Bootstrap                     100%|████████████████████| [00:00<00:00, 19808.05it/s]

,componente,invariante,max_abs_diff
0,att_gt_pos_tratamento,True,0.0
1,cohort,True,0.0
2,simple,True,0.0
3,event_k_diferente_menos_1,True,0.0
4,k_menos_1_retirado_da_banda,True,0.0
5,k_menos_1_reposto_zero_apresentacao,True,0.0


## Bootstrap final pré-especificado

A configuração abaixo é congelada antes de qualquer efeito real. Os 1.999 draws **não são executados na D15 nem na view D20B neste notebook**. `n_jobs=1` é a opção estável porque `differences==0.3.0` falhou com paralelismo Joblib/tqdm neste ambiente.

In [13]:
configuracao_final_d20b = d20a.configuracao_inferencia_final_d20b()
(OUT_DIAG / 'D20B_configuracao_inferencia.json').write_text(
    json.dumps(configuracao_final_d20b, indent=2, ensure_ascii=False), encoding='utf-8'
)
gates_d20b = d20a.gates_d20b(
    view_validada=True, workaround_validado=True, inferencia_configurada=True
)
gates_d20b_tabela = pd.DataFrame({'gate': gates_d20b.keys(), 'valor': gates_d20b.values()})
gates_d20b_tabela.to_csv(OUT_DIAG / 'D20B_gates.csv', index=False)
display(pd.DataFrame([configuracao_final_d20b]).T.rename(columns={0: 'valor'}))
display(gates_d20b_tabela)

,valor
backend,differences==0.3.0
base_period,universal
control_group,never_treated
est_method,dr
boot_iterations,1999
random_state,20260924
n_jobs,1
n_jobs_cpu_disponiveis,16
n_jobs_justificativa,1; differences 0.3.0 falha com n_jobs>1 neste ...
alpha,0.05


,gate,valor
0,D20B_CABO_FRIO_DECISAO_FECHADA,SIM
1,D20B_POPULACAO_ESTIMAVEL_128_VALIDADA,SIM
2,D20B_VIEW_BALANCEADA_VALIDADA,SIM
3,D20B_WORKAROUND_K_MENOS_1_VALIDADO,SIM
4,D20B_INFERENCIA_CONFIGURADA,SIM
5,D20B_BOOTSTRAP_FINAL_PRE_ESPECIFICADO,SIM
6,D20B_PRONTO_PARA_ESTIMACAO_REAL,SIM
7,D20B_EFEITO_REAL_ESTIMADO,NAO
8,DESENHO_CAUSAL_APROVADO,NAO


## Fechamento D20B

A decisão de Cabo Frio, a população estimável de 128 tratados, a view balanceada e o workaround de `k=-1` estão tecnicamente validados. `D20B_PRONTO_PARA_ESTIMACAO_REAL = SIM` significa apenas que estes dois bloqueios pré-estimação foram fechados; não é execução automática nem aprovação do desenho causal.

`D20B_EFEITO_REAL_ESTIMADO = NAO`  
`DESENHO_CAUSAL_APROVADO = NAO`